In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader

from tqdm.auto import tqdm

from dfm.data.salinas import (
    SALINAS_CLASS_NAMES,
    SalinasPatchDataset,
    load_salinas,
)

from dfm.models.transformer_baseline import (
    IndependentBandTransformerClassifier,
)

from dfm.training.metrics import (
    accuracy_score,
    macro_f1_score,
)

from dfm.training.profiling import count_parameters

c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

Device: cuda
GPU: NVIDIA GeForce RTX 2050
CUDA version: 11.8


In [3]:
data_dir = PROJECT_ROOT / "data" / "raw" / "salinas"

scene = load_salinas(
    data_dir,
    download=True,
)

print("Cube shape (H, W, C):", scene.cube.shape)
print("Label map shape:", scene.labels.shape)
print("Bands:", scene.bands)
print("Classes:", len(scene.class_names))

Cube shape (H, W, C): (512, 217, 204)
Label map shape: (512, 217)
Bands: 204
Classes: 16


In [4]:
outputs_dir = PROJECT_ROOT / "outputs" / "salinas"

split_path = (
    outputs_dir / "salinas_spatial_split_seed42.npz"
)

split = np.load(split_path)

train_indices = split["train_indices"]
val_indices = split["val_indices"]
test_indices = split["test_indices"]

print("Train samples:", len(train_indices))
print("Validation samples:", len(val_indices))
print("Test samples:", len(test_indices))

Train samples: 32337
Validation samples: 10952
Test samples: 10840


In [5]:
patch_size = 15

train_dataset = SalinasPatchDataset(
    scene,
    indices=train_indices,
    patch_size=patch_size,
)

val_dataset = SalinasPatchDataset(
    scene,
    indices=val_indices,
    patch_size=patch_size,
)

test_dataset = SalinasPatchDataset(
    scene,
    indices=test_indices,
    patch_size=patch_size,
)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

Train samples: 32337
Validation samples: 10952
Test samples: 10840


In [6]:
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=0,
)

print("DataLoaders created.")

DataLoaders created.


In [7]:
sample_x, sample_y = train_dataset[0]

print("Sample patch:", sample_x.shape)
print("Sample label:", sample_y)

Sample patch: (204, 15, 15)
Sample label: 7


In [12]:
# Create the independent-band Transformer
#
# Salinas:
#   - 204 spectral bands
#   - 16 classes
#   - 15x15 input patches
#
# We use patch_size=15 to match the existing
# independent-band Transformer experiment.

model = IndependentBandTransformerClassifier(
    num_classes=len(SALINAS_CLASS_NAMES),
    patch_size=15,
    embed_dim=128,
    depth=4,
    num_heads=4,
    mlp_ratio=4.0,
    dropout=0.1,
    max_bands=256,
    max_patches=4096,
).to(device)

print(model)

print(
    "Trainable parameters:",
    count_parameters(model)
)

IndependentBandTransformerClassifier(
  (patch_embed): IndependentBandPatchEmbed(
    (embed): Linear(in_features=225, out_features=128, bias=True)
    (band_embedding): Embedding(256, 128)
    (patch_embedding): Embedding(4096, 128)
  )
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (norm): LayerNorm((128,), eps=1e-05, elementwise_affine

In [13]:
# Check that the Transformer accepts our actual Salinas input

x, y = next(iter(train_loader))

print("Input:", x.shape)
print("Labels:", y.shape)

x = x.to(
    device=device,
    dtype=torch.float32,
)

with torch.no_grad():
    logits = model(x)

print("Output:", logits.shape)

Input: torch.Size([128, 204, 15, 15])
Labels: torch.Size([128])
Output: torch.Size([128, 16])


In [14]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

print("Loss:", criterion)
print("Optimizer:", optimizer)

Loss: CrossEntropyLoss()
Optimizer: AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0.0001
)


In [15]:
def train_one_epoch(model, loader):

    model.train()

    total_loss = 0.0
    total_samples = 0

    for x, y in tqdm(
        loader,
        desc="Training",
        leave=False,
    ):
        x = x.to(
            device=device,
            dtype=torch.float32,
        )

        y = y.to(
            device=device,
            dtype=torch.long,
        )

        optimizer.zero_grad(set_to_none=True)

        logits = model(x)

        loss = criterion(logits, y)

        loss.backward()

        optimizer.step()

        total_loss += (
            loss.item() * x.shape[0]
        )

        total_samples += x.shape[0]

    return total_loss / total_samples

In [16]:
@torch.no_grad()
def evaluate(model, loader):

    model.eval()

    all_true = []
    all_pred = []

    for x, y in tqdm(
        loader,
        desc="Validation",
        leave=False,
    ):
        x = x.to(
            device=device,
            dtype=torch.float32,
        )

        logits = model(x)

        pred = (
            logits
            .argmax(dim=1)
            .cpu()
            .numpy()
        )

        all_pred.append(pred)
        all_true.append(y.numpy())

    y_true = np.concatenate(all_true)
    y_pred = np.concatenate(all_pred)

    return {
        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
        "macro_f1": macro_f1_score(
            y_true,
            y_pred,
            num_classes=len(SALINAS_CLASS_NAMES),
        ),
    }

In [17]:
epochs = 30
patience = 8

best_macro_f1 = -1.0
best_epoch = None
best_state = None

epochs_without_improvement = 0

history = []

transformer_checkpoint_path = (
    outputs_dir / "transformer_spatial_best.pt"
)

print(
    "Checkpoint:",
    transformer_checkpoint_path
)

Checkpoint: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\outputs\salinas\transformer_spatial_best.pt


In [ ]:
for epoch in range(1, epochs + 1):

    train_loss = train_one_epoch(
        model,
        train_loader,
    )

    val_metrics = evaluate(
        model,
        val_loader,
    )

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_accuracy": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
    }

    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"Loss: {train_loss:.4f} | "
        f"Val Acc: {val_metrics['accuracy']:.4f} | "
        f"Val Macro-F1: {val_metrics['macro_f1']:.4f}"
    )

    if val_metrics["macro_f1"] > best_macro_f1:

        best_macro_f1 = val_metrics["macro_f1"]
        best_epoch = epoch
        epochs_without_improvement = 0

        best_state = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        }

        torch.save(
            {
                "model_state_dict": best_state,
                "best_epoch": best_epoch,
                "best_val_macro_f1": best_macro_f1,
                "class_names": SALINAS_CLASS_NAMES,
                "patch_size": patch_size,
                "bands": scene.bands,
                "seed": SEED,
            },
            transformer_checkpoint_path,
        )

        print(
            "✓ New best Transformer model — checkpoint saved"
        )

    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:

        print(
            f"Early stopping at epoch {epoch}"
        )

        break

Epoch 01 | Loss: 0.3446 | Val Acc: 0.6593 | Val Macro-F1: 0.5604
✓ New best Transformer model — checkpoint saved


Epoch 02 | Loss: 0.0487 | Val Acc: 0.8812 | Val Macro-F1: 0.7778
✓ New best Transformer model — checkpoint saved


Epoch 03 | Loss: 0.0304 | Val Acc: 0.8406 | Val Macro-F1: 0.8035
✓ New best Transformer model — checkpoint saved


Epoch 04 | Loss: 0.0267 | Val Acc: 0.8290 | Val Macro-F1: 0.7703


Epoch 05 | Loss: 0.0517 | Val Acc: 0.9049 | Val Macro-F1: 0.8305
✓ New best Transformer model — checkpoint saved


...

Epoch 07 | Loss: 0.0141 | Val Acc: 0.8656 | Val Macro-F1: 0.8079


Epoch 08 | Loss: 0.0335 | Val Acc: 0.8245 | Val Macro-F1: 0.7700


Epoch 09 | Loss: 0.0290 | Val Acc: 0.8638 | Val Macro-F1: 0.7966


Epoch 10 | Loss: 0.0370 | Val Acc: 0.8957 | Val Macro-F1: 0.8286


Training:  56%|█████▌    | 142/253 [04:38<04:21,  2.35s/it]